In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
import keras
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


In [5]:
data = pd.read_csv('Churn_Modelling.csv')

In [6]:
data = data.drop(columns = ['RowNumber', 'CustomerId', 'Surname'])
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [10]:
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns= onehot_encoder_geo.get_feature_names_out())

In [11]:
data = pd.concat([data.drop('Geography', axis = 1), geo_encoded_df], axis = 1)

X = data.drop('Exited', axis = 1)
y = data['Exited']

In [13]:
X_train,X_test,y_train,y_test = train_test_split(X,y, random_state= 42, test_size=0.2)

In [14]:
import pickle

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)


In [29]:
## Define a function to create the model and try different parameters(KerasClassifier)
def create_model(neurons = 32, layers = 1):
    model = Sequential()
    model.add(Dense(neurons, activation= 'relu', input_shape = (X_train.shape[1],)))

    for _ in range(layers -1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation = 'sigmoid'))
    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

    return model
    

In [30]:
# Create KerasClassifier
model = KerasClassifier(layers=1, neurons= 32, build_fn=create_model, epochs = 50, batch_size= 10, verbose= 0 )

In [31]:
param_grid = {
    'neurons'   : [16,32,64,128],
    'layers'    : [1,2,3],
    'epochs'    : [50,100]
}

In [32]:
grid = GridSearchCV(estimator= model, param_grid=param_grid,n_jobs = -1, cv = 3)

In [33]:
grid_result = grid.fit(X_train,y_train)

d:\D Drive Data\Github Projects\churn_ann_classification\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


In [34]:
print("Best %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Best 0.856999 using {'epochs': 50, 'layers': 1, 'neurons': 32}
